Result 1: Table 1 Reproduction

Compares Degraded/Direct/Algorithm 1/Algorithm 2 reconstructions on MNIST / CIFAR-10/ CelebA, scoring each on FID/ SSIM/ RMSE

In [1]:
from google.colab import drive
import matplotlib.pyplot as plt
drive.mount("/content/drive")
repo= "/content/drive/MyDrive/cs4782-final-project"
%cd {repo}

Mounted at /content/drive
/content/drive/MyDrive/cs4782-final-project


In [2]:
!pip install -q -r requirements.txt

DATASETS = {
    # "mnist": dict(
    #     kernel_size= 11, num_timesteps=20, kernel_std= 7.0,
    #     blur_routine="Constant", image_channels=1,
    # ),
    # "cifar10": dict(
    #     kernel_size=11, num_timesteps=50, kernel_std=0.1,
    #     blur_routine= "Special6", image_channels=3,
    # ),
    "mnist_party": dict(
        kernel_size= 11, num_timesteps=20, kernel_std= 7.0,
        blur_routine="Constant", image_channels=1,
    ),
    "mnist_gaussian_blur_focused": dict(
        kernel_size= 11, num_timesteps=20, kernel_std= 7.0,
        blur_routine="Constant", image_channels=1,
    ),
    # "celeba": dict(
    #     kernel_size=15, num_timesteps=200, kernel_std= 0.01,
    #     blur_routine="Exponential_Reflect", image_channels= 3,
    # ),

}

CHECKPOINTS= {
    # "mnist": "./checkpoints/mnist_complete.pt",
    "mnist_party": "./checkpoints/mnist_party.pt",
    "mnist_gaussian_blur_focused": "./checkpoints/mnist_gaussian_blur_focused.pt",
    # "cifar10": "./checkpoints/cifar10.pt",
    # "celeba": "./checkpoints/celeba.pt",

}

BATCH_SIZE= 64

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 62.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 90.3 MB/s eta 0:00:00


In [ ]:
import torch
from data.datasets import get_loader
from src.model import build_unet
from src.degradation import Degradation
from src.evaluate import evaluate_model

device= "cuda" if torch.cuda.is_available() else "cpu"

print("device: ", device)

def run_one(dataset_name, model_name = None):
    cfg= DATASETS[dataset_name]
    T = cfg["num_timesteps"]

    if "_" in dataset_name:
      dataset_name = dataset_name.split('_', 1)[0]
    test_loader= get_loader(dataset_name, "test", batch_size= BATCH_SIZE, num_workers= 2, shuffle= False)
    degradation= Degradation(**cfg).to(device)
    model = build_unet(dataset_name).to(device)
    if model_name is not None:
      ckpt= torch.load(CHECKPOINTS[model_name], map_location=device)
    else:
      ckpt= torch.load(CHECKPOINTS[dataset_name], map_location=device)
    # use ema weights and not raw model
    model.load_state_dict(ckpt["ema"])
    model.eval()

    return evaluate_model(model, degradation, test_loader, t=T, device= device)




ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_31036/4263771402.py", line 1, in <cell line: 0>
    import torch
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1322, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 1262, in _find_spec
  File "<frozen importlib._bootstrap_external>", line 1532, in find_spec
  File "<frozen importlib._bootstrap_external>", line 1504, in _get_spec
  File "<frozen importlib._bootstrap_external>", line 1483, in _path_importer_cache
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 2099, in showtraceback
  

In [ ]:
!pip install torch-fidelity

In [ ]:
from torchmetrics.image.fid import FrechetInceptionDistance
import torch_fidelity
# If fidelity doesn't work, run this block, restart kernel, then run again:
import sys
!{sys.executable} -m pip install -U torchmetrics
print(torch_fidelity.__version__)
import torch_fidelity
print(torch_fidelity.__file__)
fid = FrechetInceptionDistance(feature=2048)
print("FID initialized successfully")

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_31036/1048229443.py", line 1, in <cell line: 0>
    from torchmetrics.image.fid import FrechetInceptionDistance
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1322, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 1262, in _find_spec
  File "<frozen importlib._bootstrap_external>", line 1532, in find_spec
  File "<frozen importlib._bootstrap_external>", line 1504, in _get_spec
  File "<frozen importlib._bootstrap_external>", line 1483, in _path_importer_cache
OSError: [Errno 107] Transport endpoint is not connected

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/inter

In [ ]:
results= {}

for i in CHECKPOINTS:
    print(f"--- {i} ---")
    results[i] = run_one(i,i)
    print(results[i])



--- mnist_party ---
{'degraded': {'fid': 398.1429748535156, 'ssim': 0.014097359962761402, 'rmse': 0.2866146877320285}, 'direct': {'fid': 314.4747619628906, 'ssim': 0.02760329656302929, 'rmse': 0.34565480764205536}, 'alg1': {'fid': 358.55438232421875, 'ssim': 0.02909971959888935, 'rmse': 0.5448222028247482}, 'alg2': {'fid': 302.7232360839844, 'ssim': 0.2731834948062897, 'rmse': 0.2530434098555581}}
--- mnist_gaussian_blur_focused ---
{'degraded': {'fid': 398.1429748535156, 'ssim': 0.014097359962761402, 'rmse': 0.2866146877320285}, 'direct': {'fid': 385.8901062011719, 'ssim': -0.009443337097764015, 'rmse': 0.43327670312663064}, 'alg1': {'fid': 345.6249084472656, 'ssim': 0.03891201317310333, 'rmse': 0.310100633338188}, 'alg2': {'fid': 347.27960205078125, 'ssim': 0.015465078875422478, 'rmse': 0.2849038158370087}}


In [ ]:
def format_table(results):
    out = ["| Dataset | Method  | FID  | SSIM  | RMSE  |",
           "|---------|---------|------|-------|--------|"]
    for ds, res in results.items():
        for method in ["degraded", "direct", "alg1", "alg2"]:
            r= res[method]
            out.append(f"| {ds:<7} | {method:<8} | {r['fid']:6.2f} | {r['ssim']:.4f} | {r['rmse']:.4f} |")

    return "\n".join(out)

table= format_table(results)

print(table)

| Dataset | Method  | FID  | SSIM  | RMSE  |
|---------|---------|------|-------|--------|
| mnist_party | degraded | 398.14 | 0.0141 | 0.2866 |
| mnist_party | direct   | 314.47 | 0.0276 | 0.3457 |
| mnist_party | alg1     | 358.55 | 0.0291 | 0.5448 |
| mnist_party | alg2     | 302.72 | 0.2732 | 0.2530 |
| mnist_gaussian_blur_focused | degraded | 398.14 | 0.0141 | 0.2866 |
| mnist_gaussian_blur_focused | direct   | 385.89 | -0.0094 | 0.4333 |
| mnist_gaussian_blur_focused | alg1     | 345.62 | 0.0389 | 0.3101 |
| mnist_gaussian_blur_focused | alg2     | 347.28 | 0.0155 | 0.2849 |


In [ ]:
import json, os

os.makedirs("./results", exist_ok= True)

with open("./results/table1.json", "w") as f:
    json.dump(results, f, indent= 2)

with open("./results/table1.md", "w") as f:
    f.write(table)

print("saved to ./results/")

saved to ./results/
